# Sections 6 & 7 across 30 non-flaring holdout ARs — does seeding the ensemble change its error bar?

Every bootstrap ensemble in this project shares one seed, so its member spread samples
**finite-sample** variance only: which 168 frames a replicate drew. The optimization path —
LoRA init draw, dropout, data order — is held fixed, and whatever that path contributes to
model uncertainty is invisible to the spread.

`168hrs_stable_seed_diag` perturbs both at once. Arm *i* trains replicate *i* at `seed: i`, so
its across-member spread is the **total** of the two components rather than the first alone. If
the bootstrap spread were already capturing most of the model uncertainty, adding seed variation
should barely move it. If the spread grows, the published error bars are missing a component.

This notebook measures that on the holdout, not on validation, using the §6/§7 suite of
`view_holdout_nonflaring30_sections67_672hr.ipynb` over the same 30 quiescent mid-sized ARs:
boundary uncertainty, boundary error against truth, their ratio, the variance decomposition, and
calibration.

| | |
|---|---|
| **sweep** | `168hrs_stable_seed_diag` — replicates 1–10 at seeds 1–10, stabilized recipe |
| **baseline** | the shipped `168hrs` — the same ten replicates at the implicit seed 0 |
| **epoch** | best-**generalized** per member in both arms, not `best.pth` |

## Four things to hold on to before reading any number

**1. The contrast is seed AND LR schedule, not seed alone.** The clean seed-0 parent for this
sweep would have been `168hrs_stable` — same stabilized recipe (warmup 18 → cosine → clip 1.0),
differing from each arm in nothing but `seed`. `NOTES` generated its configs and it was never
trained. The only trained seed-0 ensemble on these ten replicates is the shipped `168hrs`, which
ran **constant lr 1e-3** with no warmup and no clipping. So every difference below has two
possible causes and this notebook cannot separate them. `NOTES` says the same thing in as many
words: *"AS A SEED DIAGNOSTIC IT IS CONFOUNDED."* Read a **null** result as the stronger one — if
perturbing seed *and* schedule together does not move the error bar, seed alone certainly does
not.

**2. Both arms are read at their best-generalized epoch, and that is unusual here.** Every other
inference run in this project used `best.pth`, i.e. the valid/iou argmax, which `NOTES` rule 8
shows is systematically the worst-calibrated checkpoint of its run. `epoch_select.py`
mechanizes the eight rules instead, so §7's calibration numbers are *not* comparable to the
published `672hr`/`672hr_stable` figures. §6's boundary numbers are less sensitive to this; both
are flagged where they appear.

**3. Uncertainty is not error, and they are never merged.** Member scatter measures only what the
ensemble can see. All members share the frozen `surya.366m.v1.pt` and one training distribution,
so any bias common to them is invisible to the spread — which is precisely what the ground-truth
row exists to measure.

**4. The truth masks are whole-disk.** `union_with_intersect` covers every AR in the frame, not
just the catalog's target HARP, so "non-flaring" describes the target AR and part of the error is
boundary error on its neighbours. One Surya patch is 16 px and that is the model's resolution
floor; read the tail and the across-frame spread, not the median.

Read-only: no torch, no GPU. Everything comes from the caches `analyze_holdout_sections67.py`
wrote, plus `epoch_select.py`'s selection table.

## Setup

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import yaml
from scipy import stats

if "." not in sys.path:
    sys.path.insert(0, ".")
import ensemble_viz as ev      # palette + raster helpers, shared with the other viewers
import false_color as fcolor   # SURYA_PATCH_PX

ev.apply_rcparams()

SWEEP_ROOT = "/efs/bathomas/ar_seg_bootstrap/168hrs_stable_seed_diag_infer_holdout_nonflaring30"
# The shipped seed-0 ensemble over the *same* 30 ARs at the *same* epoch policy. Read per frame
# rather than quoted, so every number below is a paired difference instead of two headlines.
BASE_ROOT = "/efs/bathomas/ar_seg_bootstrap/168hrs_gen_infer_holdout_nonflaring30"
# The two 672-sample cohorts, for cross-regime scale only. Read from their own caches rather
# than retyped; they were selected by best-IoU, so they are context, not a comparison.
REF_ROOTS = {
    "672hr": "/efs/bathomas/ar_seg_bootstrap/672hr_infer_holdout_nonflaring30",
    "672hr_stable": "/efs/bathomas/ar_seg_bootstrap/672hr_stable_infer_holdout_nonflaring30",
}
EPOCH_CSV = "/efs/bathomas/ar_seg_bootstrap/168hrs_epoch_selection.csv"
CATALOG = "./assets/surya-bench-ar-segmentation/holdout_ar_nonflaring_30_from_test.csv"
SCALERS_PATH = "./assets/scalers.yaml"

PATCH_PX = fcolor.SURYA_PATCH_PX
MIN_N = 1000                 # reliability floor, as in the single-frame notebook
TRAIN_TRUE_FRAC = 0.00643    # training-era validation positive fraction
CHANNEL = "aia94"

frames = pd.read_csv(os.path.join(SWEEP_ROOT, "metrics_frame.csv"))
members = pd.read_csv(os.path.join(SWEEP_ROOT, "metrics_member_frame.csv"))
bins = pd.read_csv(os.path.join(SWEEP_ROOT, "calibration_bins.csv"))
dhist = pd.read_csv(os.path.join(SWEEP_ROOT, "distance_hist.csv"))
base_frames = pd.read_csv(os.path.join(BASE_ROOT, "metrics_frame.csv"))
base_members = pd.read_csv(os.path.join(BASE_ROOT, "metrics_member_frame.csv"))
base_bins = pd.read_csv(os.path.join(BASE_ROOT, "calibration_bins.csv"))
base_dhist = pd.read_csv(os.path.join(BASE_ROOT, "distance_hist.csv"))
catalog = pd.read_csv(CATALOG)
epochs = pd.read_csv(EPOCH_CSV)
scalers = yaml.safe_load(open(SCALERS_PATH))
ref = {k: pd.read_csv(os.path.join(v, "metrics_frame.csv")) for k, v in REF_ROOTS.items()}

MEMBERS = sorted(members.member.unique())
N = len(MEMBERS)
NF = len(frames)

print(f"sweep    {NF} frames x {N} members = {len(members)} member-frame rows")
print(f"baseline {len(base_frames)} frames x {base_members.member.nunique()} members "
      f"= {len(base_members)} rows")

# Every downstream aggregate assumes a complete balanced design; a partial inference run would
# otherwise be averaged silently.
assert len(members) == NF * N, f"expected {NF * N} sweep member-frame rows, got {len(members)}"
assert len(base_members) == NF * N, f"baseline incomplete: {len(base_members)}"

for tag, f in [("sweep", frames), ("baseline", base_frames)]:
    bad = f[f.mean_check_abs_diff > 1e-3]
    empty = f[f.empty_boundary_members.fillna("").astype(str).str.strip().ne("")]
    print(f"{tag:9s} fp16 mean-check failures: {len(bad) if len(bad) else 'none'};  "
          f"frames where a member predicted nothing: {len(empty) if len(empty) else 'none'}")

# The pairing is verifiable, not asserted: ground-truth positive fraction is a property of the
# masks alone, so it MUST be bit-identical across the two runs.
pair = frames.merge(base_frames, on="harpnum", suffixes=("", "_old"))
assert len(pair) == NF, f"paired on {len(pair)} of {NF} ARs"
dtruth = (pair.truth_frac - pair.truth_frac_old).abs().max()
print(f"\npaired on {len(pair)}/{NF} ARs;  max |truth_frac difference| = {dtruth:.3e}  "
      f"({'identical, pairing is real' if dtruth == 0 else 'NOT IDENTICAL -- pairing broken'})")
assert dtruth == 0

## 1. What was run, and at which epoch

The epoch is the one thing this comparison had to choose, so it is reported before any metric.
`epoch_select.py` applied `NOTES`' eight rules to each member's own curve: the **window**
runs from the last epoch that took new high ground in IoU by more than 2 jitter σ to the first
post-floor epoch whose validation loss is more than 10 % above its floor, and the pick is the
earliest epoch inside that window still within 2 σ of the run's peak IoU.

Two things to notice in the table. The sweep's `boot01` sits on a **10-epoch / 1680-step cosine**
while `boot02`–`boot10` are on 4-epoch / 672-step ones — the budget drift `NOTES` flagged and
nobody fixed before the run — so epoch indices are not equivalent across its members and
per-member selection is mandatory rather than merely tidy. And the baseline's jitter σ is far
larger on several members: at constant lr its operating point thrashes epoch to epoch, which is
why two of its picks land on epoch 0.

In [ ]:
sel = epochs.copy()
sel["arm"] = sel.arm.map({"168hrs_stable_seed_diag": "sweep", "168hrs": "baseline"})
view = sel[["arm", "member", "replicate", "n_epochs", "window_lo", "window_hi", "chosen_epoch",
            "best_iou_epoch", "loss_floor_epoch", "sigma", "iou_chosen", "iou_best",
            "iou_deficit_sigma", "vloss_excess_pct", "regime"]].copy()
view.columns = ["arm", "member", "replicate", "n ep", "w_lo", "w_hi", "pick", "bIoU", "flr",
                "sigma", "IoU", "IoU max", "def/s", "+%flr", "diagnosis"]
for arm in ("sweep", "baseline"):
    print(f"--- {arm}")
    print(view[view.arm == arm].drop(columns="arm").to_string(
        index=False, float_format=lambda x: f"{x:.4f}"))
    print()

for arm in ("sweep", "baseline"):
    a = sel[sel.arm == arm]
    print(f"{arm:9s} validation IoU at the chosen epoch  "
          f"{a.iou_chosen.mean():.4f} +/- {a.iou_chosen.std(ddof=1):.4f}   "
          f"[{a.iou_chosen.min():.4f}, {a.iou_chosen.max():.4f}]")
    print(f"{'':9s} best.pth would sit "
          f"{(a.best_iou_epoch - a.loss_floor_epoch).mean():.1f} epochs past the loss floor, "
          f"costing {a.vloss_excess_at_best_iou_pct.mean():+.1f}% validation loss")
    print(f"{'':9s} median jitter sigma {a.sigma.median():.4f}   "
          f"max {a.sigma.max():.4f} ({a.loc[a.sigma.idxmax(), 'member']})")

# The epoch each run actually used is recorded in the prob_*.nc attrs, so the caches can confirm
# the tables rather than being trusted to match them.
for tag, m, arm in [("sweep", members, "sweep"), ("baseline", base_members, "baseline")]:
    got = m.groupby("member").epoch.first().to_dict()
    want = dict(zip(sel[sel.arm == arm].member, sel[sel.arm == arm].chosen_epoch))
    print(f"\n{tag:9s} epochs in the cache match the selection table: {got == want}")
    assert got == want, f"{tag}: cache {got} != table {want}"

### The cohort

The column that matters most is **truth AR fraction**: these 30 ARs are the mid-sized quiescent
control, and §6's absolute scale depends on how much AR is actually in each frame.

In [ ]:
cohort = frames.merge(
    catalog[["harpnum", "area_km2_million", "n_flares", "max_goes_class", "n_euv_events"]],
    on="harpnum", how="left",
).sort_values("truth_frac", ascending=False)

tbl = cohort[["harpnum", "frame", "area_km2_million", "n_flares", "n_euv_events", "truth_frac",
              "truth_density_vs_train", "soft_iou", "boundary_assd_px",
              "boundary_error_assd_px", "boundary_error_over_uncertainty", "brier", "bias"]]
tbl.columns = ["HARP", "frame", "area Mkm2", "fl", "EUV", "truth frac", "x train", "IoU",
               "UNC px", "ERR px", "ratio", "Brier", "bias"]
print(tbl.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\ntruth AR fraction: mean {frames.truth_frac.mean():.5f}  "
      f"median {frames.truth_frac.median():.5f}  "
      f"[{frames.truth_frac.min():.5f}, {frames.truth_frac.max():.5f}]  "
      f"= {frames.truth_density_vs_train.median():.2f}x the training-era rate "
      f"({TRAIN_TRUE_FRAC:.5f}) at the median")

## 2. §6 — how well located is the boundary, and does seeding widen it?

Three numbers carry this section:

- **UNC** — mean ASSD of the ten members' own $p_m = 0.5$ boundaries about the ensemble-mean
  boundary. What the ensemble thinks it knows.
- **ERR** — the same measurement against the ground-truth mask. How wrong it is.
- **ratio** — ERR / UNC. **1.0 would mean the spread is an honest error bar.**

Reported as mean ± *between-frame* sd. Two poolings of the ratio answer different questions: the
mean of 30 per-frame ratios weights every AR equally, while mean-ERR / mean-UNC is the
cohort-level ratio and is lower whenever one frame has a tiny UNC.

**The seed question is the UNC row.** If the optimization path carries variance the bootstrap
misses, the sweep's UNC should exceed the baseline's.

In [ ]:
def describe(series, fmt="{:.3f}"):
    v = pd.to_numeric(series, errors="coerce").dropna()
    return (f"{fmt.format(v.mean())} +/- {fmt.format(v.std(ddof=1))}   "
            f"median {fmt.format(v.median())}   "
            f"[{fmt.format(v.min())}, {fmt.format(v.max())}]")


ROWS = [
    ("UNCERTAINTY  ASSD (px)", "boundary_assd_px", "{:.2f}"),
    ("  between-member sd (px)", "boundary_assd_sd_px", "{:.2f}"),
    ("  median (px)", "boundary_median_px", "{:.2f}"),
    ("  mean HD95 (px)", "boundary_hd95_px", "{:.1f}"),
    ("ERROR  ASSD (px)", "boundary_error_assd_px", "{:.2f}"),
    ("  median (px)", "boundary_error_median_px", "{:.2f}"),
    ("  HD95 (px)", "boundary_error_hd95_px", "{:.1f}"),
    ("ERROR / UNCERTAINTY", "boundary_error_over_uncertainty", "{:.2f}"),
    ("sigma_p on the boundary", "boundary_sigma_prob", "{:.4f}"),
    ("soft-mean IoU", "soft_iou", "{:.4f}"),
    ("member IoU sd", "member_iou_sd", "{:.4f}"),
]

for tag, f in [("SWEEP (seeds 1-10)", frames), ("BASELINE (seed 0, shipped)", base_frames)]:
    print(f"=== {tag}")
    print(f"{'':<34}{'mean +/- sd':>22}   median / range")
    print("-" * 78)
    for label, col, fmt in ROWS:
        print(f"{label:<34}{describe(f[col], fmt)}")
    print()

for tag, f in [("sweep", frames), ("baseline", base_frames)]:
    unc, err = f.boundary_assd_px.mean(), f.boundary_error_assd_px.mean()
    r = f.boundary_error_over_uncertainty
    print(f"{tag:9s} cohort-level mean-ERR/mean-UNC {err / unc:.2f}x   "
          f"mean of per-frame ratios {r.mean():.2f}x   median {r.median():.2f}x   "
          f"frames above 1.0: {int((r > 1).sum())}/{NF}   above 2.0: {int((r > 2).sum())}/{NF}")

print("\ncross-regime scale, for context only (best-IoU epochs, 672-sample replicates):")
for k, f in ref.items():
    print(f"  {k:14s} UNC {f.boundary_assd_px.mean():5.2f} px   "
          f"ERR {f.boundary_error_assd_px.mean():5.2f} px   "
          f"ratio {f.boundary_error_over_uncertainty.mean():.2f}x (mean of per-frame)")

### The paired test — the only one that can resolve this

The between-frame sd of the ratio is larger than any plausible shift between the two arms, so
unpaired cohort means resolve nothing. The 30 ARs are identical in both runs, so every row below
is a **Wilcoxon signed-rank test on the 30 per-AR differences**.

`n up` counts ARs where the sweep's value exceeds the baseline's. For UNC, "up" is the direction
that would mean seeding exposes variance the bootstrap missed.

In [ ]:
def paired(col, fmt="{:.3f}"):
    "One row: sweep mean, baseline mean, the difference, and Wilcoxon on the difference."
    new = pd.to_numeric(pair[col], errors="coerce")
    old = pd.to_numeric(pair[f"{col}_old"], errors="coerce")
    d = (new - old).dropna()
    _, p = stats.wilcoxon(d)
    return (f"{fmt.format(new.mean()):>10}{fmt.format(old.mean()):>10}"
            f"{fmt.format(d.mean()):>11}{fmt.format(d.median()):>11}"
            f"{int((d > 0).sum()):>5}/{len(d)}{p:>11.2e}"
            f"{'  *' if p < 0.05 else ''}")


print(f"{'':<30}{'sweep':>10}{'seed 0':>10}{'mean d':>11}{'median d':>11}"
      f"{'n up':>8}{'Wilcoxon':>11}")
print("-" * 92)
for label, col, fmt in [
    ("UNCERTAINTY ASSD (px)", "boundary_assd_px", "{:.2f}"),
    ("member ASSD sd (px)", "boundary_assd_sd_px", "{:.2f}"),
    ("member HD95 (px)", "boundary_hd95_px", "{:.1f}"),
    ("ERROR ASSD (px)", "boundary_error_assd_px", "{:.2f}"),
    ("error HD95 (px)", "boundary_error_hd95_px", "{:.1f}"),
    ("ERROR / UNCERTAINTY", "boundary_error_over_uncertainty", "{:.2f}"),
    ("sigma_p on the boundary", "boundary_sigma_prob", "{:.4f}"),
    ("soft-mean IoU", "soft_iou", "{:.4f}"),
    ("soft precision", "soft_precision", "{:.4f}"),
    ("soft recall", "soft_recall", "{:.4f}"),
    ("member IoU sd", "member_iou_sd", "{:.4f}"),
    ("ensemble pred fraction", "soft_pred_frac", "{:.4f}"),
]:
    print(f"{label:<30}{paired(col, fmt)}")
print("\n* = p < 0.05")

d_unc = pair.boundary_assd_px - pair.boundary_assd_px_old
d_ratio = pair.boundary_error_over_uncertainty - pair.boundary_error_over_uncertainty_old
print(f"\n  ARs where seeding WIDENED the spread: {int((d_unc > 0).sum())}/{NF}   "
      f"relative change in UNC, median {100 * (d_unc / pair.boundary_assd_px_old).median():+.1f}%")
print(f"  ARs whose ratio improved (fell toward 1): {int((d_ratio < 0).sum())}/{NF}")
print(f"  ARs above 1.0 under both arms: "
      f"{int(((pair.boundary_error_over_uncertainty > 1) & (pair.boundary_error_over_uncertainty_old > 1)).sum())}/{NF}")
print(f"  worst single member-frame ASSD:  sweep {members.assd_px.max():.1f} px   "
      f"baseline {base_members.assd_px.max():.1f} px")
print(f"  member-frames above 100 px:      sweep {int((members.assd_px > 100).sum())}/{len(members)}   "
      f"baseline {int((base_members.assd_px > 100).sum())}/{len(base_members)}")

### The paired comparison as a chart

Left: each AR's uncertainty under both arms, against the line of equality — points off the
diagonal are ARs the seeding moved. Right: the same for error against truth. If seeding added a
variance component, the left panel's cloud sits above the diagonal and the right panel's does
not move.

Log axes on both: UNC spans more than a decade across this cohort and a linear axis collapses
most of it onto the origin.

In [ ]:
fig, (axl, axr) = plt.subplots(1, 2, figsize=(13.5, 5.8))
fig.suptitle(f"{NF} non-flaring holdout ARs  |  168 hr ensembles, seeds 1-10 vs seed 0  |  "
             "paired per AR", fontsize=13, fontweight="bold", color=ev.INK, y=1.0)

for ax, (col, name, better) in zip((axl, axr), [
    ("boundary_assd_px", "boundary UNCERTAINTY, member scatter (px)", "wider"),
    ("boundary_error_assd_px", "boundary ERROR against truth (px)", "worse"),
]):
    x = pair[f"{col}_old"].values
    y = pair[col].values
    lim = [min(x.min(), y.min()) * 0.8, max(x.max(), y.max()) * 1.25]
    ax.plot(lim, lim, color=ev.MUTED, lw=1.2, zorder=1)
    ax.scatter(x, y, s=70, color=ev.SERIES_1, alpha=0.85, zorder=3,
               edgecolors=plt.rcParams["figure.facecolor"], linewidths=1.5)
    ax.axvline(PATCH_PX, color=ev.INK_2, lw=0.9, ls=(0, (4, 3)), zorder=2)
    ax.axhline(PATCH_PX, color=ev.INK_2, lw=0.9, ls=(0, (4, 3)), zorder=2)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(f"seed 0 (shipped 168hrs)  —  {name}")
    ax.set_ylabel(f"seeds 1-10 (sweep)")
    ax.set_title(name, fontsize=11, color=ev.INK, loc="left")
    n_up = int((y > x).sum())
    ax.annotate(f"above the line ({better}): {n_up}/{NF}\ndashed line = one Surya patch",
                (0.03, 0.97), xycoords="axes fraction", va="top", fontsize=9, color=ev.MUTED)
    ev.recessive(ax, grid_axis="both")

plt.tight_layout()
plt.show()

### Is any one member reliably tighter than the others?

**Kendall's $W$** is the concordance of the 30 per-frame member rankings: 1.0 means every frame
orders the members identically, 0 means the orderings are unrelated. A high $W$ says member
identity is a real, transferable property; a low one says per-frame ranking is noise.

On the stabilized 672 hr ensemble $W$ collapsed to 0.104 once its broken checkpoints were gone —
the earlier 0.331 had been three members being reliably bad. The question here is whether
*seeding* a member changes its character, which the right-hand table answers by pairing each
arm's member against its own replicate.

In [ ]:
def per_member_table(m):
    t = m.groupby("member").agg(
        epoch=("epoch", "first"), val_iou=("val_iou", "first"),
        assd_mean=("assd_px", "mean"), assd_sd=("assd_px", "std"),
        assd_min=("assd_px", "min"), assd_max=("assd_px", "max"),
        median_px=("median_px", "mean"), hd95=("hd95_px", "mean"), iou=("iou", "mean"),
    ).sort_values("assd_mean")
    wide = m.pivot(index="frame", columns="member", values="assd_px")[list(t.index)]
    ranks = wide.rank(axis=1)
    t["mean_rank"] = ranks.mean()
    t["rank_sd"] = ranks.std(ddof=1)
    return t, wide, ranks


def kendall_w(ranks):
    "k frames rating n members; W = 12 S / (k^2 (n^3 - n))."
    k, n = ranks.shape
    rank_sums = ranks.sum(axis=0).values
    s = ((rank_sums - rank_sums.mean()) ** 2).sum()
    return 12 * s / (k ** 2 * (n ** 3 - n))


for tag, m in [("SWEEP", members), ("BASELINE", base_members)]:
    t, wide, ranks = per_member_table(m)
    out = t[["epoch", "val_iou", "assd_mean", "assd_sd", "assd_min", "assd_max", "median_px",
             "hd95", "iou", "mean_rank", "rank_sd"]].copy()
    out.columns = ["ep", "val IoU", "ASSD px", "sd", "min", "max", "median", "HD95", "IoU",
                   "rank", "rank sd"]
    print(f"=== {tag}  (sorted by mean ASSD about the ensemble boundary)")
    print(out.to_string(float_format=lambda x: f"{x:.3f}"))
    w = kendall_w(ranks)
    chi2 = w * ranks.shape[0] * (ranks.shape[1] - 1)
    print(f"  Kendall's W = {w:.3f}   chi2 = {chi2:.1f} on {ranks.shape[1] - 1} df, "
          f"p = {stats.chi2.sf(chi2, ranks.shape[1] - 1):.2e}")
    print(f"  tightest {t.index[0]} {t.assd_mean.iloc[0]:.2f} px   "
          f"loosest {t.index[-1]} {t.assd_mean.iloc[-1]:.2f} px   "
          f"spread {t.assd_mean.iloc[-1] / t.assd_mean.iloc[0]:.1f}x\n")

# Does a replicate keep its character when reseeded? Pair member-for-member on mean ASSD.
sw = members.groupby("member").assd_px.mean()
bs = base_members.groupby("member").assd_px.mean()
common = sorted(set(sw.index) & set(bs.index))
rho, p_rho = stats.spearmanr(sw[common], bs[common])
print(f"per-member mean ASSD, sweep vs baseline on the same replicate: "
      f"Spearman {rho:+.3f} (p {p_rho:.3f}, n={len(common)})")
print("  high = replicate identity survives reseeding; low = member character is not a property")
print("  of the replicate at all.")

### Variance decomposition — is the member spread an error bar, or is the frame?

A one-way random-effects split of the 300 per-member ASSD values, with frame as the grouping
factor:

$$\sigma^2_{\text{total}} = \sigma^2_{\text{between frames}} + \sigma^2_{\text{within frame (members)}}$$

The **intraclass correlation** $\text{ICC} = \sigma^2_b / (\sigma^2_b + \sigma^2_w)$ is the
fraction of all variation attributable to *which AR you are looking at*. Both 672-sample cohorts
landed at 0.19, i.e. 81 % of the variation is members disagreeing — the quantity an error bar is
supposed to be about. Whether seeding changes that split is the question.

In [ ]:
def icc_oneway(wide):
    m_, k_ = wide.shape[1], wide.shape[0]
    grand = wide.values.mean()
    frame_means = wide.mean(axis=1)
    ms_between = m_ * ((frame_means - grand) ** 2).sum() / (k_ - 1)
    ms_within = ((wide.values - frame_means.values[:, None]) ** 2).sum() / (k_ * (m_ - 1))
    var_between = max((ms_between - ms_within) / m_, 0.0)
    return dict(grand=grand, ms_between=ms_between, ms_within=ms_within,
                sigma_between=np.sqrt(var_between), sigma_within=np.sqrt(ms_within),
                icc=var_between / (var_between + ms_within))


for tag, m in [("sweep", members), ("baseline", base_members)]:
    _, wide, _ = per_member_table(m)
    d = icc_oneway(wide)
    print(f"=== {tag}  ({wide.shape[0]} frames x {wide.shape[1]} members)")
    print(f"  grand mean ASSD                 {d['grand']:.3f} px")
    print(f"  sigma_between (frame identity)  {d['sigma_between']:.3f} px   "
          f"({100 * d['icc']:.0f} % of variance)")
    print(f"  sigma_within  (member spread)   {d['sigma_within']:.3f} px   "
          f"({100 * (1 - d['icc']):.0f} %)")
    print(f"  ICC                             {d['icc']:.3f}\n")

print("sigma_within is the quantity a bootstrap error bar is made of; comparing it between the")
print("two arms is the variance-component question stated in pixels rather than in IoU.")

### Method A vs Method B

$\sigma_r = \sigma_p / |\nabla \bar p|$ (A, the linearized level-set displacement) and the
empirical ASSD (B, the headline) rest on different assumptions. Across the 672-sample cohorts
they did **not** agree — A is a *local* linearization, structurally blind to a detached island or
a whole region the ensemble missed, which is exactly what inflates B. It is reported here for
continuity and because its two assumptions are measured rather than asserted, not as a
cross-check. **B is the headline.**

In [ ]:
for tag, f in [("SWEEP", frames), ("BASELINE", base_frames)]:
    c = f[["boundary_assd_px", "sigma_r_mean_px", "sigma_r_median_px", "sigma_r_p90_px",
           "within_patch_range", "seam_bias_ratio"]].copy()
    c["rel_diff_pct"] = 100 * (c.sigma_r_mean_px - c.boundary_assd_px) / c.boundary_assd_px
    print(f"=== {tag}")
    print(f"{'':<34}{'mean +/- sd':>22}   median / range")
    print("-" * 78)
    for label, col, fmt in [
        ("Method B  ASSD (px)", "boundary_assd_px", "{:.2f}"),
        ("Method A  sigma_r mean (px)", "sigma_r_mean_px", "{:.2f}"),
        ("Method A  sigma_r p90 (px)", "sigma_r_p90_px", "{:.1f}"),
        ("A - B, as % of B", "rel_diff_pct", "{:+.1f}"),
        ("within-patch range of p-bar", "within_patch_range", "{:.3f}"),
        ("patch-seam gradient bias (x)", "seam_bias_ratio", "{:.2f}"),
    ]:
        print(f"{label:<34}{describe(c[col], fmt)}")
    r, p_r = stats.pearsonr(c.boundary_assd_px, c.sigma_r_mean_px)
    within20 = int((c.rel_diff_pct.abs() < 20).sum())
    print(f"  A vs B across frames: Pearson r = {r:+.3f} (p {p_r:.3f});  "
          f"{within20}/{len(c)} frames agree within 20 %\n")

### The whole distribution, not just its mean

Every boundary-to-boundary distance from all 30 frames, pooled. Read any point as *"this fraction
of boundary pixels lie within x px"*. Members in grey, ground truth in orange — the gap between
them **is** the gap between what the ensemble thinks it knows and how wrong it is.

Built from the per-frame histograms the analysis script accumulated (0.25 px bins out to 64 px,
log-spaced beyond), so this is the exact pooled distribution over ~$10^8$ distances rather than a
sample of it. One patch is an exact bin edge, so the "within one patch" figures are read off a
boundary rather than interpolated.

In [ ]:
def pooled_ecdf(df):
    "Summed counts -> (upper bin edge, cumulative fraction, total)."
    g = df.groupby("bin_hi", as_index=False)["count"].sum().sort_values("bin_hi")
    total = g["count"].sum()
    return g.bin_hi.values, np.cumsum(g["count"].values) / total, total


fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), sharey=True)
fig.suptitle(f"{NF} non-flaring ARs  |  how far boundary pixels lie from the ensemble-mean "
             "boundary", fontsize=13, fontweight="bold", color=ev.INK, y=1.02)

for ax, (tag, dh) in zip(axes, [("seeds 1-10 (sweep)", dhist), ("seed 0 (shipped)", base_dhist)]):
    is_truth = dh.series == "ground truth"
    for member in sorted(dh.loc[~is_truth, "series"].unique()):
        x, y, _ = pooled_ecdf(dh[dh.series == member])
        ax.plot(x, y, color=ev.MUTED, lw=1.2, alpha=0.55, zorder=2)
    xt, yt, n_t = pooled_ecdf(dh[is_truth])
    ax.plot(xt, yt, color=ev.SERIES_2, lw=2.4, zorder=4)
    ax.axvline(PATCH_PX, color=ev.INK_2, lw=1.0, zorder=1)
    ax.set_xscale("log")
    ax.set_xlabel("distance from the ensemble-mean boundary (px)")
    ax.set_title(tag, fontsize=11, color=ev.INK, loc="left")
    frac_m = np.interp(PATCH_PX, *pooled_ecdf(dh[~is_truth])[:2])
    frac_t = np.interp(PATCH_PX, xt, yt)
    ax.annotate(f"within one patch:\n members {100 * frac_m:.1f}%\n truth {100 * frac_t:.1f}%",
                (0.03, 0.95), xycoords="axes fraction", va="top", fontsize=9, color=ev.MUTED)
    ev.recessive(ax, grid_axis="both")
axes[0].set_ylabel("fraction of boundary pixels within x")
axes[0].annotate("one Surya patch", (PATCH_PX, 0.04), xytext=(7, 0),
                 textcoords="offset points", fontsize=8.5, color=ev.INK_2)
plt.tight_layout()
plt.show()

for tag, dh in [("sweep", dhist), ("baseline", base_dhist)]:
    is_truth = dh.series == "ground truth"
    xm, ym, nm = pooled_ecdf(dh[~is_truth])
    xt, yt, nt = pooled_ecdf(dh[is_truth])
    print(f"{tag:9s} pooled over {nm / 1e6:.1f} M member distances and {nt / 1e6:.2f} M truth "
          f"distances;  within one patch: members {100 * np.interp(PATCH_PX, xm, ym):.1f}%, "
          f"truth {100 * np.interp(PATCH_PX, xt, yt):.1f}%")

### Does the ratio track mask density?

The obvious confounder: if error-over-uncertainty simply grows with how much AR is in the frame,
then any difference between the arms could be a density artifact. It is not — the cohort is
identical in both arms by construction — but the null is worth re-checking at this epoch policy,
because it is the one that licenses reading the ratio as a property of the *ensemble*.

Spearman rather than Pearson: the question is monotonicity, and both quantities are long-tailed.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
fig.suptitle(f"{NF} non-flaring ARs  |  what does error / uncertainty depend on?",
             fontsize=13, fontweight="bold", color=ev.INK, y=1.02)

for ax, (col, xlabel, logx) in zip(axes, [
    ("truth_frac", "ground-truth AR fraction of the frame", True),
    ("soft_iou", "ensemble soft-mean IoU", False),
    ("boundary_sigma_prob", r"$\sigma_p$ on the boundary", False),
]):
    for f, color, tag in [(base_frames, ev.MUTED, "seed 0"),
                          (frames, ev.SERIES_1, "seeds 1-10")]:
        rho, p_s = stats.spearmanr(f[col], f.boundary_error_over_uncertainty)
        ax.scatter(f[col], f.boundary_error_over_uncertainty, s=55, color=color, alpha=0.85,
                   zorder=3, edgecolors=plt.rcParams["figure.facecolor"], linewidths=1.2,
                   label=f"{tag}: rho {rho:+.2f} (p {p_s:.2f})")
    ax.axhline(1.0, color=ev.INK_2, lw=1.2, zorder=1)
    if logx:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.legend(frameon=False, fontsize=8.5, loc="upper right")
    ev.recessive(ax, grid_axis="both")
axes[0].set_ylabel("error / uncertainty")
axes[0].annotate("honest error bar", (axes[0].get_xlim()[0], 1.0), xytext=(4, 4),
                 textcoords="offset points", fontsize=8.5, color=ev.INK_2)
plt.tight_layout()
plt.show()

## 3. §7 — Calibration, over 30 frames

Bin pixels by $\bar p$ and compare against the observed ground-truth frequency. On the diagonal
is perfect; **below is over-confident, above is under-confident**.

**A different axis from §2, and the two must not be merged.** §2 asked whether the member spread
is an honest error bar on the boundary's *position*. This asks whether $\bar p$ is an honest
frequency for a pixel's *class*.

**Restricted to pixels where at least one member fired.** The $k = 0$ population is ~95 % of each
frame and would set the curve's left end on its own; its own AR content is reported separately,
because that is where the ensemble's misses hide.

The pooled curve sums the raw per-frame bin accumulators ($n$, $n_{\text{true}}$,
$\sum \bar p$), so it is the exact curve over all 30 frames' pixels — not an average of 30
per-frame curves, which would weight a sparse frame the same as a dense one.

**This is the section most affected by the epoch policy.** Both arms sit at or near their
validation-loss floor by construction, so these numbers should be *better* calibrated than any
published `best.pth` figure in this project, and are not comparable to them.

In [ ]:
def pool(b):
    p = b.groupby(["bin_lo", "bin_hi"], as_index=False)[["n", "n_true", "sum_p"]].sum()
    p["mean_p"] = p.sum_p / p.n
    p["observed"] = p.n_true / p.n
    p["reliable"] = p.n >= MIN_N
    return p


def summarize(p, f, tag):
    n_tot = int(p.n.sum())
    mean_p = float(p.sum_p.sum() / n_tot)
    observed = float(p.n_true.sum() / n_tot)
    ok = p.reliable & p.n.gt(0)
    w = p.n[ok] / p.n[ok].sum()
    ece = float((w * (p.observed[ok] - p.mean_p[ok]).abs()).sum())
    # Exact pooled Brier: the n-weighted mean of the per-frame Briers.
    brier = float((f.brier * f.n_binned).sum() / f.n_binned.sum())
    print(f"{tag:9s} pixels {n_tot / 1e6:7.2f} M   mean p-bar {mean_p:.4f}   "
          f"observed {observed:.4f}   bias {mean_p - observed:+.4f}   "
          f"Brier {brier:.4f}   ECE {ece:.4f}")
    return dict(pooled=p, mean_p=mean_p, observed=observed, ece=ece, brier=brier)


sw_cal = summarize(pool(bins), frames, "sweep")
bs_cal = summarize(pool(base_bins), base_frames, "baseline")

print("\nreliability table, sweep (pooled over all 30 frames):")
p = sw_cal["pooled"]
show = p.assign(bin=lambda d: d.bin_lo.map("{:.1f}".format) + "-" + d.bin_hi.map("{:.1f}".format))
print(show[["bin", "n", "n_true", "mean_p", "observed", "reliable"]].to_string(
    index=False, float_format=lambda x: f"{x:.4f}"))

print("\nthe k = 0 population, where the misses hide:")
for tag, f in [("sweep", frames), ("baseline", base_frames)]:
    print(f"  {tag:9s} {100 * f.quiet_frac.mean():.1f} % of each frame has no member firing, "
          f"and it still holds {100 * f.missed_ar_frac.mean():.1f} +/- "
          f"{100 * f.missed_ar_frac.std(ddof=1):.1f} % of that frame's true AR")

### Is the direction of the miscalibration consistent, and did seeding move it?

The pooled curve can hide cancelling frames. If the sign of the bias is the same on nearly every
AR, the miscalibration is a property of the ensemble; if mixed, the pooled number is an average
of two different behaviours and no single threshold correction applies.

In [ ]:
print(f"{'':<34}{'mean +/- sd':>22}   median / range")
print("-" * 78)
for tag, f in [("SWEEP", frames), ("BASELINE", base_frames)]:
    print(f"=== {tag}")
    for label, col, fmt in [
        ("Brier (per frame)", "brier", "{:.4f}"),
        ("bias (p-bar - observed)", "bias", "{:+.4f}"),
        ("ECE", "ece", "{:.4f}"),
        ("mean p-bar where >=1 fired", "mean_p", "{:.4f}"),
    ]:
        print(f"{label:<34}{describe(f[col], fmt)}")
    n_under = int((f.bias < 0).sum())
    _, p_w = stats.wilcoxon(f.bias)
    print(f"  under-confident (bias < 0): {n_under}/{NF}   over-confident: {NF - n_under}/{NF}   "
          f"Wilcoxon vs 0: p = {p_w:.2e}  -> sign is "
          f"{'CONSISTENT' if p_w < 0.05 else 'not consistent'}\n")

print(f"{'':<30}{'sweep':>10}{'seed 0':>10}{'mean d':>11}{'median d':>11}"
      f"{'n up':>8}{'Wilcoxon':>11}")
print("-" * 92)
for label, col, fmt in [
    ("Brier", "brier", "{:.4f}"),
    ("bias (p-bar - observed)", "bias", "{:+.4f}"),
    ("|bias|", "abs_bias", "{:.4f}"),
    ("ECE", "ece", "{:.4f}"),
    ("mean p-bar", "mean_p", "{:.4f}"),
    ("missed AR fraction at k=0", "missed_ar_frac", "{:.4f}"),
]:
    if col == "abs_bias":
        pair["abs_bias"] = pair.bias.abs()
        pair["abs_bias_old"] = pair.bias_old.abs()
    print(f"{label:<30}{paired(col, fmt)}")
print("\n* = p < 0.05;  'n up' counts ARs where the sweep's value exceeds the baseline's")

### The pooled reliability curve

The 30 per-frame curves sit behind the pooled one in grey, so the spread is visible rather than
averaged away. Bin counts are in the lower panel on a log scale, **not** a second y-axis: a bin
holding $10^3$ pixels and one holding $10^7$ must not be drawn as equals.

In [ ]:
fig, (ax, axc) = plt.subplots(2, 1, figsize=(9, 8.4), sharex=True,
                              gridspec_kw={"height_ratios": [2.4, 1], "hspace": 0.12})
fig.suptitle(f"{NF} non-flaring holdout ARs  |  reliability of the ensemble mean",
             fontsize=13, fontweight="bold", color=ev.INK, y=0.95)

ax.plot([0, 1], [0, 1], color=ev.MUTED, lw=1.2, zorder=2)
ax.annotate("perfect calibration", (0.72, 0.72), xytext=(0, -14), textcoords="offset points",
            ha="center", fontsize=9, color=ev.MUTED, rotation=38, rotation_mode="anchor")

for frame_id, g in bins.groupby("frame"):
    g = g[g.n >= MIN_N]
    if len(g) > 1:
        ax.plot(g.sum_p / g.n, g.n_true / g.n, color=ev.MUTED, lw=1.0, alpha=0.4, zorder=3)

for cal, color, tag, ls in [(bs_cal, ev.INK_2, "seed 0 (shipped)", (0, (5, 2))),
                            (sw_cal, ev.SERIES_1, "seeds 1-10 (sweep)", "solid")]:
    p = cal["pooled"]
    ok = p.reliable & p.n.gt(0)
    ax.plot(p.mean_p[ok], p.observed[ok], color=color, lw=2.4, ls=ls, zorder=5,
            label=f"{tag}   bias {cal['mean_p'] - cal['observed']:+.4f}, ECE {cal['ece']:.4f}")
    ax.scatter(p.mean_p[ok], p.observed[ok], s=45, color=color, zorder=6,
               edgecolors=plt.rcParams["figure.facecolor"], linewidths=1.5)
    axc.step(p.bin_hi, p.n, where="pre", color=color, lw=1.6, ls=ls)

ax.set_ylabel(r"observed ground-truth frequency")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=9, loc="upper left")
ax.annotate("above the diagonal = under-confident", (0.03, 0.62), xycoords="axes fraction",
            fontsize=8.5, color=ev.MUTED)
ev.recessive(ax, grid_axis="both")

axc.set_yscale("log")
axc.set_xlabel(r"ensemble mean probability $\bar p$   (pixels where at least one member fired)")
axc.set_ylabel("pixels in bin")
axc.axhline(MIN_N, color=ev.SERIES_2, lw=1.0, ls=(0, (4, 3)))
axc.annotate(f"reliability floor, n = {MIN_N}", (0.02, MIN_N), xytext=(0, 5),
             textcoords="offset points", fontsize=8.5, color=ev.SERIES_2)
ev.recessive(axc, grid_axis="y")
plt.show()

## 4. Three exemplars

The aggregate needs something concrete under it. The AR with the **lowest**, the **median** and
the **highest** error-over-uncertainty in the sweep, each with the $\sigma_r$ map on its own most
contested window.

$\sigma_r$ is recomputed here from the cached $\bar p$ and $\sigma_p$ rather than stored: it
costs a second and would have added 67 MB per frame to the cache. Drawn only on the transition
band $0.1 < \bar p < 0.9$ — outside it, $\sigma_p / |\nabla \bar p|$ divides a near-zero
numerator by a near-zero denominator, which is numerical noise with no boundary to be the
uncertainty *of*. Values are clipped at one patch rather than rescaled, so the colour means the
same thing on all three panels.

In [ ]:
ordered = frames.sort_values("boundary_error_over_uncertainty").reset_index(drop=True)
picks = [("lowest", ordered.iloc[0]), ("median", ordered.iloc[NF // 2]),
         ("highest", ordered.iloc[-1])]


def load_channel(frame_dir, channel):
    "One channel in display units: undoes the standardization only, as infer_file does."
    with xr.open_dataset(os.path.join(frame_dir, "channels.nc"), engine="h5netcdf") as ds:
        normalized = ds[channel].values.astype(np.float32)
    s = scalers[channel]
    return normalized * (s["std"] + s["epsilon"]) + s["mean"]


fig, axes = plt.subplots(2, 3, figsize=(16.5, 11.4))
fig.suptitle(r"boundary position uncertainty $\sigma_r$ on the most contested window"
             "\nlowest / median / highest error-over-uncertainty of the 30  (sweep)",
             fontsize=13, fontweight="bold", color=ev.INK, y=1.0)

for col, (tag, row) in enumerate(picks):
    frame_dir = os.path.join(SWEEP_ROOT, row.frame)
    y0, x0, y1, x1 = int(row.win_y0), int(row.win_x0), int(row.win_y1), int(row.win_x1)
    with xr.open_dataset(os.path.join(frame_dir, "ensemble_mean.nc"), engine="h5netcdf") as ds:
        p_mean = ds.ar_probability_mean.values[y0:y1, x0:x1].astype(np.float32)
        p_sd = ds.ar_probability_sd.values[y0:y1, x0:x1].astype(np.float32)
        truth = ds.ground_truth.values[y0:y1, x0:x1].astype(np.float32)

    img = load_channel(frame_dir, CHANNEL)[y0:y1, x0:x1]
    vmin, vmax = ev.channel_clim(CHANNEL, img)
    ax = axes[0, col]
    ax.imshow(img, cmap=ev.channel_cmap(CHANNEL), vmin=vmin, vmax=vmax, origin="lower")
    ax.contour(p_mean, levels=[0.5], colors=[ev.CONTOUR[0.5]], linewidths=1.4)
    ax.contour(truth, levels=[0.5], colors=[ev.MASK_FILL], linewidths=1.2, linestyles="dashed")
    ax.set_title(f"{tag}: HARP {int(row.harpnum)}  {row.frame}\n"
                 f"UNC {row.boundary_assd_px:.1f} px   ERR {row.boundary_error_assd_px:.1f} px   "
                 f"ratio {row.boundary_error_over_uncertainty:.2f}x",
                 fontsize=10, color=ev.INK)
    ax.set_xticks([]); ax.set_yticks([])

    gy, gx = np.gradient(p_mean)
    grad = np.hypot(gy, gx)
    band = (p_mean > 0.1) & (p_mean < 0.9) & (grad > 0)
    sigma_r = np.full_like(p_mean, np.nan)
    sigma_r[band] = np.clip(p_sd[band] / grad[band], 0, PATCH_PX)
    ax = axes[1, col]
    im = ax.imshow(sigma_r, cmap=ev.SEQ, vmin=0, vmax=PATCH_PX, origin="lower")
    ax.set_title(r"$\sigma_r = \sigma_p / |\nabla \bar p|$  on $0.1 < \bar p < 0.9$"
                 f"\nmedian {np.nanmedian(sigma_r):.1f} px over "
                 f"{int(band.sum()) / 1e3:.0f}k px", fontsize=10, color=ev.INK)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03, label="px (clipped at one patch)")

axes[0, 0].annotate(f"solid = ensemble $\\bar p$ = 0.5\ndashed = ground truth\n{CHANNEL}",
                    (0.03, 0.97), xycoords="axes fraction", va="top", fontsize=8.5,
                    color="white")
plt.tight_layout()
plt.show()

## 5. What this cohort says

In [ ]:
d_unc = pair.boundary_assd_px - pair.boundary_assd_px_old
d_err = pair.boundary_error_assd_px - pair.boundary_error_assd_px_old
d_ratio = pair.boundary_error_over_uncertainty - pair.boundary_error_over_uncertainty_old
_, p_unc = stats.wilcoxon(d_unc)
_, p_err = stats.wilcoxon(d_err)
_, p_ratio = stats.wilcoxon(d_ratio)

sw_w = icc_oneway(per_member_table(members)[1])
bs_w = icc_oneway(per_member_table(base_members)[1])

print("THE SEED QUESTION, in one block")
print("-" * 78)
print(f"  boundary UNCERTAINTY   sweep {frames.boundary_assd_px.mean():6.2f} px   "
      f"seed 0 {base_frames.boundary_assd_px.mean():6.2f} px   "
      f"paired d {d_unc.mean():+.2f} px (median {d_unc.median():+.2f}), p {p_unc:.3f}")
print(f"  boundary ERROR         sweep {frames.boundary_error_assd_px.mean():6.2f} px   "
      f"seed 0 {base_frames.boundary_error_assd_px.mean():6.2f} px   "
      f"paired d {d_err.mean():+.2f} px (median {d_err.median():+.2f}), p {p_err:.3f}")
print(f"  ERROR / UNCERTAINTY    sweep {frames.boundary_error_over_uncertainty.mean():6.2f}x   "
      f"seed 0 {base_frames.boundary_error_over_uncertainty.mean():6.2f}x   "
      f"paired d {d_ratio.mean():+.2f}, p {p_ratio:.3f}")
print(f"  member spread sigma_w  sweep {sw_w['sigma_within']:6.2f} px   "
      f"seed 0 {bs_w['sigma_within']:6.2f} px      "
      f"ICC {sw_w['icc']:.3f} vs {bs_w['icc']:.3f}")
print(f"  validation IoU         sweep "
      f"{sel[sel.arm == 'sweep'].iou_chosen.mean():.4f} +/- "
      f"{sel[sel.arm == 'sweep'].iou_chosen.std(ddof=1):.4f}   seed 0 "
      f"{sel[sel.arm == 'baseline'].iou_chosen.mean():.4f} +/- "
      f"{sel[sel.arm == 'baseline'].iou_chosen.std(ddof=1):.4f}")
print()
print(f"  Both arms still have error well above uncertainty: "
      f"{int((frames.boundary_error_over_uncertainty > 1).sum())}/{NF} and "
      f"{int((base_frames.boundary_error_over_uncertainty > 1).sum())}/{NF} frames above 1.0x.")
print("  Whatever seeding does or does not add, it does not close that gap -- the gap is bias,")
print("  and all 20 arms share the frozen surya.366m.v1.pt backbone.")
print()
print("  CAVEAT, restated because it bounds every line above: the baseline differs from the")
print("  sweep in LR SCHEDULE as well as seed, because 168hrs_stable was never trained. A null")
print("  result here is the strong reading; a difference cannot be attributed to the seed.")

### Reading it

*Filled in from the executed output above — the numbers in the block, not a restatement of the
hypothesis.*